# Assessment 8

## Instructions

1. Import the data located at this link. It has information on people infected with dengue at the district level for 2015 to 2021.
2. Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: Use this code.
3. Use geopandas to plot the number of **cases in 2021 by the district** using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile.
4. Use geopandas to plot the number of **cases in 2021 by the province** using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the province level.
5. Use geopandas to plot the number of **cases by the department for all the years** using subplots. Every subplot for each year. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level.
6. Use geopandas to plot the number of **cases by the department for all 2021 quarters** using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.

### 0) Installing and importing necessary packages

In [ ]:
#pip install geopandas

In [ ]:
import os
import pandas as pd
import numpy as np
import csv

import geopandas as gpd
import matplotlib.pyplot as plt
from geopandas import GeoDataFrame
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm

### 1) Import  the data located at this link. It has information on people infected with dengue at the district level for 2015 to 2021.

In [ ]:
# Keeps the column "Ubigeo" as string and helps choosing the type of each column
dtypes = {'Ubigeo': str, 'Semana':int, 'Año': int, 'Departamento': str, 'Provincia': str, 'Distrito': str, 'Eventos o daños': str} 

# Imports csv
df = pd.read_csv(r'../../_data/data_dengue_peru.csv', dtype = dtypes)

# Converts the column "Casos" in a int variable
df['Casos'] = df['Casos'].fillna(-9999).astype(str) # the Na´s value converts to "-9999"
df['Casos'] = df['Casos'].str.replace(',', '').astype(float).astype(int)

df.head(5)

In [ ]:
# Checking the columns' types
print("Columns' types ")
print(df.dtypes)

print('----------------------------')

# Checking NA's in the df
print("Are there any Na's in the df?")
isna_columns = df.isna().sum()
print(isna_columns)

print('----------------------------')

# Checking the data from "Casos"
print("Total observations for each registration in 'Casos'")
print(df.groupby('Casos').size())

### 2) Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: Use this code.

For exercise 2, we did't use the hint because we import the column "Ubigeo" as a string variable. This simplifies the process for getting the id code for departments (two first Ubigeo's number) and provinces (four first Ubigeo's number). 

This data frame will help to solve exercise 5

In [ ]:
# Creating new columns and taking similar names from similar columns in shapefile 
df['CCDD'] = df['Ubigeo'].str[:2] 
df['IDPROV'] = df['Ubigeo'].str[:4]
df = df.rename({'Ubigeo':'UBIGEO'}, axis =1 )

df.head(5)

### 3) Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile.

#### a) Creating a subdata of dengue cases in 2021


This data frame will help to solve exercises 3, 4 and 6

In [ ]:
# Creating subdata for year 2021 only
df_21 = df [ df['Año'] == 2021]

# Replacing -9999 for NaN in the column 'Casos'
df_21['Casos'] = df_21['Casos'].replace(-9999, np.nan).astype(float)

df_21.head(5)

#### b) Creating an aux subdata for solving exercise 3

In [ ]:
# List for keeping columns
aux_colums = ['UBIGEO', 'Departamento', 'Provincia', 'Distrito']

# Collapse by "Ubigeo" and sum column "Casos"
df_21_dist = df_21.groupby(aux_colums)['Casos'].sum()

# Resetting the index for the new df
df_21_dist = df_21_dist.reset_index()

df_21_dist.head(5)

In [ ]:
# Checking NA's in the df
print("Are there any Na's in the df for exercise 3?")
isna_columns = df_21_dist.isna().sum()
print(isna_columns)

#### c) Importing the shapefile

In [ ]:
# Loading the shapefile
shapefile_path = (r'../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')
maps_dist = gpd.read_file(shapefile_path)

maps_dist.head(5)

In [ ]:
# Checking unique values in "UBIGEO" shapefile column
print("The column 'UBIGEO' has only unique values:", maps_dist['UBIGEO'].is_unique)
print("The number of unique values is:" , maps_dist['UBIGEO'].unique().size)

# Select only relevant columns
maps_dist = maps_dist[['UBIGEO', 'geometry']]
maps_dist.head(5)

#### d) Merging the necessary data frames for plotting the number of cases in 2021 by the district 

In [ ]:
ex_1 = pd.merge(maps_dist, df_21_dist, how="inner", on="UBIGEO")
ex_1

#### e) Plotting Dengue infections in 2021 at distric level

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

maps_dist.plot(ax=ax,color ='lightgrey',
          linewidth=0.8,
          linestyle='-',
          edgecolor='gray',
          legend = True)

ex_1.plot( column='Casos', cmap='Reds', 
          linewidth=0.8, 
          ax=ax,
          linestyle='-',
          edgecolor='gray',
          legend = True)

# Aggregates the title of the map
ax.set_title('Number of Dengue Cases in 2021 by District')
ax.set_axis_off()

# Create a combined legend
handles, labels = ax.get_legend_handles_labels()
handles.append(plt.Line2D([0], [0], color='lightgrey', marker='s', markersize=10))
labels.append('NA Values')
handles.append(plt.Line2D([0], [0], color='gray', lw=2))
labels.append('District Boundaries')

ax.legend(handles, labels, loc='lower left')

### 4. Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. For this task, you will have to aggregate shapefiles at the province level.

In [ ]:
# Loading the shapefile of distric again
shapefile_path = (r'../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')
maps_prov = gpd.read_file(shapefile_path)

maps_prov.head(5)

In [ ]:
# We aggregate shapefiles at the province level
maps_province = maps_prov.dissolve( by = 'IDPROV', as_index=False )
maps_province

In [ ]:
# Checking unique values in "IDPROV" shapefile column
print("The column 'IDPROV' has only unique values:", maps_province['IDPROV'].is_unique)
print("The number of unique values is:" , maps_province['IDPROV'].unique().size)

In [ ]:
# List for keeping columns that we will use
aux_colums = ['IDPROV', 'Provincia']

# Collapse by "IDPROV" and sum column "Casos"
df_21_prov = df_21.groupby(aux_colums)['Casos'].sum()

# Resetting the index for the new df
df_21_prov = df_21_prov.reset_index()

df_21_prov

In [ ]:
# We merge the df of the shapefiles with the df of dengue cases to plot the number of cases in 2021 by province on a map
dengue_province = pd.merge(maps_province, df_21_prov, how="inner", on="IDPROV")
dengue_province

In [ ]:
# Plotting Dengue infections in 2021 at province level
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

maps_province.plot(ax=ax,color ='lightgrey',
          linewidth=0.8,
          linestyle='-',
          edgecolor='gray',
          legend = True)

dengue_province.plot( column='Casos', cmap='Reds', 
          linewidth=0.8, 
          ax=ax,
          linestyle='-',
          edgecolor='gray',
          legend = True)

# Aggregates the title of the map
ax.set_title('Number of Dengue Cases in 2021 by Province')
ax.set_axis_off()

# Create a combined legend
handles, labels = ax.get_legend_handles_labels()
handles.append(plt.Line2D([0], [0], color='lightgrey', marker='s', markersize=10))
labels.append('NA Values')
handles.append(plt.Line2D([0], [0], color='gray', lw=2))
labels.append('Province Boundaries')

ax.legend(handles, labels, loc='lower left')

### 5) Use geopandas to plot the number of **cases by the department for all the years** using subplots. Every subplot for each year. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level.

In [ ]:
#We read a folder, and we apply the function, and create a new data "departamento_shp"
departamento_shp = gpd.read_file( r'../../_data/INEI_LIMITE_DEPARTAMENTAL')
#Now we have to modify the name of the DEPNAME column to be able to join this data later
departamento_shp = departamento_shp.rename({'NOMBDEP':'Departamento'}, axis = 1)

In [ ]:
#We see the data created before
df.head(10)

In [ ]:
#We see the years of interest
data_casos_dengue_anual = set(df.Año)
data_casos_dengue_anual

In [ ]:
# We created a loop to see the number of cases by departament for all year.

data_casos_dengue_anual = list(data_casos_dengue_anual)
data_casos_dengue_anual.sort()

for año in data_casos_dengue_anual:
    casos_dengue_2021 = df[df["Año"] == año]
    casos_dengue_departamento= casos_dengue_2021.groupby(["Departamento"]).sum().reset_index()
    casos_dengue_departamento_2021 = casos_dengue_departamento.merge(departamento_shp, how = "right", on = "Departamento")
    casos_dengue_departamento_2021 = casos_dengue_departamento_2021[['Departamento', 'Casos', 'geometry']]
    casos_dengue_departamento_2021_gdf = GeoDataFrame(casos_dengue_departamento_2021)
    
    plot = casos_dengue_departamento_2021_gdf.plot( column='Casos', cmap='Oranges', 
                                     figsize=(20, 20),
                                     linestyle='-',
                                     edgecolor='black',
                                     legend = "True",
                                     missing_kwds= dict(color = "#E5E5E5"))
    plt.title(f"Casos de Dengue en {año} por departamento", fontsize=20)

### 6) Use geopandas to plot the number of **cases by the department for all 2021 quarters** using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.

In [ ]:
# grouping data by department and quarter
df_21['Quarter'] = pd.cut(df_21['Semana'], bins=[1, 14, 27, 40, 53], labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_21_dept_quarter = df_21.groupby(['Departamento', 'Quarter'])['Casos'].sum().reset_index()
df_21_dept_quarter.head(10)

In [ ]:
# We aggregate shapefiles at the department level
maps_department = maps_province.dissolve(by='CCDD', as_index=False)
maps_department

In [ ]:
# Checking unique values in "CCDD" shapefile column
print("The column 'CCDD' has only unique values:", maps_department['CCDD'].is_unique)
print("The number of unique values is:" , maps_department['CCDD'].unique().size)

In [ ]:
# Shapefile Preparation
maps_dept = maps_department[['CCDD', 'NOMBDEP', 'geometry']]
# Change the name of NOMBDEP column to Departamento
maps_dept = maps_dept.rename(columns={'NOMBDEP': 'Departamento'})
maps_dept .head(5)

In [ ]:
# Data Combination
ex_4 = pd.merge(maps_dept, df_21_dept_quarter, how="inner", on="Departamento")
ex_4.head(10)

In [ ]:
# Creating subplots
fig, axs = plt.subplots(2, 2, figsize=(15, 12), subplot_kw={'aspect': 'equal'})
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

for i, quarter in enumerate(quarters):
    ax = axs.flatten()[i]

    # Plot the map of Peru by department
    maps_dept.plot(ax=ax, color='lightgrey', linewidth=0.8, linestyle='-', edgecolor='gray', legend=True)

    # Filter data for the current quarter
    ex_4_quarter = ex_4[ex_4['Quarter'] == quarter]

    # Plot map for current quarter with categorical legend
    cmap = ListedColormap(['#fee0d2', '#fc9272', '#de2d26', '#a50f15', '#6a3d9a'])
    bounds = [0, 500, 1000, 1500, 2000, 2500]
    norm = BoundaryNorm(bounds, cmap.N)

    ex_4_quarter.plot(column='Casos', cmap=cmap, linewidth=0.8, ax=ax, edgecolor='grey', legend=True, legend_kwds={'label': f'Casos - {quarter}'}, norm=norm)

    # Axis and Title Settings
    ax.set_title(f'Número de casos en {quarter} 2021 por Departamento')
    ax.set_axis_off()

    # Add legend for values NaN
    handles, labels = ax.get_legend_handles_labels()
    handles.append(plt.Line2D([0], [0], color='lightgrey', marker='s', markersize=10))
    labels.append('NA Values')
    handles.append(plt.Line2D([0], [0], color='gray', lw=2))
    labels.append('Departament Boundaries')

    ax.legend(handles, labels, loc='lower left')

# Adjust the layout and display the graph
plt.tight_layout()
plt.show()